**example_2: the same notebook run on another session**, `sub-F06_ses-CPFE-F06-S1` (same mouse and lens, the 5-min pre-exposure to the chamber, 7 days before the S3 memory test used in `example.ipynb`). Code and parameters are unchanged; the text and the interpretation were written for S3.

This is a simple notebook, build over a day or so to explore the processing of neural calcium channel imagery (something that's now to me).

In [ ]:
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf, params
from caiman.summary_images import correlation_pnr

import imageio.v3 as iio
from IPython.display import Video

import caiman as cm
import numpy as np
import cv2

In [ ]:
from pathlib import Path

In [ ]:
files = sorted(Path('data/sub-F06/sub-F06_ses-CPFE-F06-S1_ophys').glob('*.avi'))

In [ ]:
fname = files[0]

In [ ]:
m = cm.load(fname)
ms = m.resize(0.5, 0.5, 1)

# Field of view: the GRIN lens projects a circular image; outside the inscribed circle of the frame there is no signal.
# Applied after the spatial high-pass (so the blur never sees an artificial edge) and to CaImAn's components after the fit.
yy, xx = np.mgrid[:ms.shape[1], :ms.shape[2]]
fov = np.hypot(yy - (ms.shape[1] - 1) / 2, xx - (ms.shape[2] - 1) / 2) <= ms.shape[1] / 2   # r <= 150 px at 300x300

In [ ]:
lo, hi = np.percentile(ms, [1, 99.5])
u8 = (np.clip((ms - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

In [ ]:
b = iio.imwrite("<bytes>", u8, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=b, embed=True, mimetype="video/mp4")

With these images, we're supposed to see neurons firing, but on the raw adata above, this isn't visible at all. It's more like a mess of haze, with lots of movement, and hard to discern signature, plus a strong constant background. Let's try to fix this, by first removing the low spatial frenquency information.

In [ ]:
low = np.array([cv2.GaussianBlur(frame, (45, 45), 4.5) for frame in ms[4:]])
high = ms[4:] - low
high[:, ~fov] = 0          # outside the lens circle: no information

In [ ]:
lo, hi = np.percentile(high, [1, 99.9])
u8h = (np.clip((high - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
b = iio.imwrite("<bytes>", u8h, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=b, embed=True, mimetype="video/mp4")

Low spatial frequency data is removed by subtracting it from the raw data, and the image looks more like a "brain" now, I think I can start to see neurons firing. However, there's a lot of sot noise there.Since this is a quick demo and I don't care about having the best possible extraction, I'll adress it using temporal smoothing.

The dark haloes are an artefact of the high-pass filter.

Now to remove the shot noise. I do this with a simple low pass filter in the time domain.

In [ ]:
from scipy import ndimage

In [ ]:
high_slow = ndimage.uniform_filter1d(high, 5, axis=0)

In [ ]:
lo, hi = np.percentile(high_slow, [1, 99.9])
u8s = (np.clip((high_slow - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
b = iio.imwrite("<bytes>", u8s, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=b, embed=True, mimetype="video/mp4")

And here it is, the movie from above, simply averaged in the time axis over five frames, or 167ms at 30fps. Averaging is a bit of a naive way of dealing with shot noise, and using say time correlations of the pixels over time would work better, but then again I just want to see neurons firing now.

There are also a few other things we can start to see: if looks like we have much better resolution and SNR on axis, and that the edges of the field of view are quite distorted, with a "fisheye" effect of some sort. i don't know the exact optical setup, but I'd guess it's a function of the lens that imposes a havy vignetting, and i would exepect a large amout of field curvature as well.

In [ ]:
high_slow_flat = high_slow / np.maximum(low[4:].mean(axis=0)[np.newaxis,:,:], 1)

In [ ]:
lo, hi = np.percentile(high_slow_flat, [1, 99.9])
u8s = (np.clip((high_slow_flat - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
b = iio.imwrite("<bytes>", u8s, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=b, embed=True, mimetype="video/mp4")

In [ ]:
base = ndimage.grey_opening(high_slow, size=(299, 1, 1))   # 299 frames ≈ 10 s, along time only
act = high_slow - base

In [ ]:
lo, hi = np.percentile(act, [1, 99.9])
u8a = (np.clip((act - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
ba = iio.imwrite("<bytes>", u8a, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=ba, embed=True, mimetype="video/mp4")

This is the result of a time wise morphological opening over a ten second window. It takes the windowed minimum over 300 frames, then applies a local time wise maximum over the same window. This kills some of the systematic drift, and only selects transients.

Activations pop up better there, but the little sideways jerks make the image jitter, and that can't be good for a pixel level thresholding based method. Let's go back to the high passed image, and try CaImAn's built in motion correction function (warnings are simply an internal CaImAn warning, they don't affect the results).

In [ ]:
mc = MotionCorrect([str(fname)], max_shifts=(35, 35), gSig_filt=(8, 8),
                   pw_rigid=False, border_nan='copy')
mc.motion_correct(save_movie=True)
shifts = np.array(mc.shifts_rig)

In [ ]:
mms = cm.load(mc.mmap_file[0]).resize(0.5, 0.5, 1)

In [ ]:
low_mc = np.array([cv2.GaussianBlur(frame, (45, 45), 4.5) for frame in mms[4:]])
high_mc = mms[4:] - low_mc
high_mc[:, ~fov] = 0       # outside the lens circle: no information
high_slow_flat_mc = high_slow / np.maximum(low_mc[4:].mean(axis=0)[np.newaxis,:,:], 1)
high_slow_mc = ndimage.uniform_filter1d(high_mc, 5, axis=0)
base_mc = ndimage.grey_opening(high_slow_mc, size=(299, 1, 1))   # 299 frames ≈ 10 s, along time only
act_mc = high_slow_mc - base_mc

In [ ]:
base_neg = ndimage.grey_opening(-high_slow_mc, size=(299, 1, 1))   # 299 frames ≈ 10 s, along time only
act_neg = -high_slow_mc - base_neg

In [ ]:
# Side by side, same contrast stretch: left = no motion correction, right = rigid NoRMCorre
both = np.concatenate([act, act_mc], axis=2)
lo, hi = np.percentile(both, [1, 99.9])
u8b = (np.clip((both - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
bb = iio.imwrite("<bytes>", u8b, extension=".mp4", fps=30, codec="libx264", quality=6, macro_block_size=1)
Video(data=bb, embed=True, mimetype="video/mp4")

With the motion corrected movie on the right, and the non corected one on the left, we can see the difference it makes. Now let's write a simple, naive pipeline to try to detect activations.

Now let's look at the processed activation movie again. The dynamic range is pretty compressed, so I'll use a square root to stretch the display.

In [ ]:
lo, hi = np.percentile(act_mc**.5, [1, 99.9])
u8a = (np.clip((act_mc**.5 - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
ba = iio.imwrite("<bytes>", u8a, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=ba, embed=True, mimetype="video/mp4")

Now we can see activations clearly, as well as some other poerturbations, such as blood pulsating in vessels. We could run a simple, max based extraction:

In [ ]:
peak = act_mc.max(axis=0)
peak_fl = act_neg.max(axis=0)
thr = np.percentile(peak[fov], 95)     # threshold from inside the lens circle only
pk = np.argwhere((peak == ndimage.maximum_filter(peak, size=7)) & (peak > thr))
pk_fl = np.argwhere((peak_fl == ndimage.maximum_filter(peak_fl, size=7)) & (peak_fl > thr))

from matplotlib import pyplot as plt
plt.imshow(peak)
plt.plot(pk[:,1],pk[:,0],'wx')

That does work, but looking at the movie again, I'd like to be able to do a few things:

- Firstly, shot noise is visible in the processed movie, and I ould like to be able to evaluate the noise level for each pixel. That would allow me to set a threshold based on this estimate, and only detect when a pixel gets hotter than a peak.
- Activation events follow a specific sequence: a blob of pixels brighten with a fast ramp up, and then decay slowly. I would like a criterion than catches this specific sequence, with say a set number of continguous pixels brightening by a set amount over a set time, then decaying at a specific rate, and to flag this as an activation
- Following from the previous point, I also would like to extract a time signal: detect when, and not only where neurons activate.

For the noise, let's first look at the brightest maximum used to find the peaks and see if we can isolate the the dead times, between activations:

In [ ]:
max_pos = pk[-1]
calm_pos = [170,150]
act_ref = ndimage.uniform_filter1d(act_mc, 61, axis=0)

fig, ax = plt.subplots((2))
ax[0].imshow(peak**.5,cmap='Greys_r')
ax[0].set_xlim(50,250)
ax[0].set_ylim(50,250)
ax[0].plot(max_pos[1],max_pos[0],'xb')
ax[0].plot(calm_pos[1],calm_pos[0],marker='x',color='orange')
ax[1].plot(act_mc[:,max_pos[0],max_pos[1]],label='Detected Maximum',linestyle='--')
ax[1].plot(act_mc[:,calm_pos[0],calm_pos[1]],label='Manually selected calm region',linestyle='--')
ax[1].legend(frameon=False,ncol=2)
ax[1].set_xlim(0,len(act_mc))
ax[1].set_ylim(0,act_mc[:,max_pos[0],max_pos[1]].max()*1.3)
ax[1].plot(act_ref[:,max_pos[0],max_pos[1]],color='tab:blue')
ax[1].plot(act_ref[:,calm_pos[0],calm_pos[1]],color='orange',linestyle='--')

The sold line respresents the two second average of each pixels, and I'll estimate the noise from the second decile of this value, thus:

In [ ]:
calm = act_ref <= np.percentile(act_ref, 20, axis=0)
resid = np.where(calm, act_mc - act_ref, np.nan) 
noise = np.nanstd(resid, axis=0) 

In [ ]:
plt.imshow(noise,cmap="Greys_r")

Now, let's use this estimated noise and a bit of morphology again to find activations. The rul is simple: a positive varation over 2 times the noise level over a patch of a minimum size 5x5, of 25 contiguous pixels, for 5 frames in a row. This is a simple rule, meant to detect activations over time.

In [ ]:
act_diff = np.diff(act_mc,axis=0)
rising = act_diff > .7 * noise
rise = ndimage.binary_erosion(rising, structure=np.ones((5, 2, 2), bool))

In [ ]:
u8a = (rise * 255).astype(np.uint8)
ba = iio.imwrite("<bytes>", u8a, extension=".mp4", fps=30,
                codec="libx264", quality=7, macro_block_size=1)
Video(data=ba, embed=True, mimetype="video/mp4")

This shows activations as they happen, with a ~3 frames lag (due to time domain erosion). 

In [ ]:
onsets = (np.diff((rise > 0).astype(np.int8), axis=0) == 1).sum(0)
sm = ndimage.gaussian_filter(onsets.astype(np.float32), 1.5)
pk_rise = np.argwhere((sm == ndimage.maximum_filter(sm, size=7)) & (onsets > 0))

In [ ]:
plt.imshow((np.diff((rise > 0).astype(np.int8), axis=0) == 1).sum(0)); plt.colorbar()

In [ ]:
coords = np.argwhere(rise.any(axis=0))

In [ ]:
from matplotlib import pyplot as plt
plt.imshow(peak)
plt.plot(coords[:,1],coords[:,0],'wx')

What looks like activations appear, but some noise makes it through.

Now this rough noise estimate can serve as an adative threshold to evaluate how strong an activation event is. To do so, let's use use a simple criterion: 5 consecutive samples where the variation is over twice the noise level, over a radius of 4 pixels.

## CaImAn CNMF-E on the motion-corrected chunk

Same chunk, ×2 downsampled, start-up frames dropped. Parameters are the authors' (stored in the NWB, `processing/ophys/processing_parameters`). Pixel-valued ones are halved for the downsampling: gSig 4 → 2, gSiz 17 → 9, patch half-size rf 40 → 20, patch overlap stride 25 → 12. Ratios are kept (ring_size_factor 1.5). The authors computed the background at 2× spatial downsampling (ssub_B 2), so on data already downsampled ×2 the equivalent is ssub_B 1. I Will not claipm to understand what all the paramters mean, I simply lifted them from the files to try and reproduce their run.

In [ ]:
# CNMF-E wants a C-order memmap; this also downsamples x2 and drops the 4 start-up frames
fname_new = cm.save_memmap([mc.mmap_file[0]], base_name='cnmfe_S1_', order='C',
                           resize_fact=(0.5, 0.5, 1), remove_init=4)
Yr, dims, T = cm.load_memmap(fname_new)
images = np.reshape(Yr.T, [T] + list(dims), order='F')

In [ ]:
opts = params.CNMFParams(params_dict={
    'fr': 29.78, 'decay_time': 1.0, 'p': 1,            # frame rate, GCaMP6s decay (s), AR(1) deconvolution
    'method_init': 'corr_pnr', 'K': None,              # CNMF-E seeding from correlation x PNR, no fixed cell count
    'gSig': (2, 2), 'gSiz': (9, 9), 'center_psf': True,  # cell radius and patch size (px at 300x300)
    'min_corr': 0.9, 'min_pnr': 10,                    # seed thresholds (authors' values)
    'ring_size_factor': 1.5, 'ssub_B': 1,              # ring background: radius = 1.5 x gSiz (authors')
    'nb': 0, 'nb_patch': 0, 'low_rank_background': None,
    'rf': 20, 'stride': 12, 'only_init': True,         # patches: half-size 20 px, overlap 12 px (authors' 40, 25 halved)
    'ssub': 1, 'tsub': 1, 'merge_thr': 0.8,
    'normalize_init': False, 'update_background_components': True,
    'method_deconvolution': 'oasis', 'del_duplicates': True, 'border_pix': 0,
})

In [ ]:
cnm = cnmf.CNMF(n_processes=1, params=opts, dview=None)
cnm.fit(images)          # ~100 s on this chunk (patches run one after another without a cluster)
# Same field-of-view mask for CaImAn: keep only components whose centre of mass lies inside the lens circle
com_px = np.clip(np.round(cm.base.rois.com(cnm.estimates.A, *dims)).astype(int), 0, np.array(dims) - 1)
inside = fov[com_px[:, 0], com_px[:, 1]]
print(f"components inside the lens circle: {inside.sum()}/{len(inside)}")
cnm.estimates.select_components(idx_components=np.flatnonzero(inside))
cnm.estimates.A.shape, cnm.estimates.C.shape   # (pixels, components), (components, frames)

In [ ]:
import matplotlib.pyplot as plt
from caiman.utils.visualization import plot_contours
cn, pnr = correlation_pnr(images, gSig=2, swap_dim=False)
C = cnm.estimates.C[:8]
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
plt.sca(ax[0]); plot_contours(cnm.estimates.A, cn * pnr, display_numbers=False)
ax[1].plot(C.T / C.max(axis=1) + np.arange(len(C)))
ax[1].set_xlabel('frame'); ax[1].set_ylabel('component (normalised C)');

Now the interesting question: how well did my on naive pipeline actually locate activations? Let's plot the countours in white as before, and my own centroids in red.

In [ ]:
from caiman.utils.visualization import get_contours
conts = get_contours(cnm.estimates.A, dims, thr=0.2, thr_method='max')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for i, c in enumerate(conts):
    ax.plot(*c['coordinates'].T, 'k', lw=0.8, label="CaImAn's detected contours" if i == 0 else None)
ax.plot(pk_rise[:, 1], pk_rise[:, 0], 'rx', label="My own naive pipeline, centroids")
ax.set_xlim(0, dims[1]); ax.set_ylim(dims[0], 0); ax.set_aspect('equal')
ax.legend()

And the comparison. Of course, an afternoon of tinkering on data I hadn't seen before doesn't agree perfectly with CaImAn, a tool years in the making, by domain level experts, but I'm pretty happy with how much it agrees with CNMF-E as specified. For the numbers:

In [ ]:
from scipy.spatial import cKDTree
centres = cm.base.rois.com(cnm.estimates.A, *dims)   # (row, col), same order as pk
d_pk, _ = cKDTree(centres).query(pk)   # each of my peaks -> nearest CaImAn centre
d_cm, _ = cKDTree(pk).query(centres)   # each CaImAn centre -> nearest of my peaks
r = 3   # match radius, px (9.6 um at 3.2 um/px)
print(f"precision: {(d_pk <= r).sum()}/{len(pk)} of my peaks are within {r} px of a CaImAn centre")
print(f"recall:    {(d_cm <= r).sum()}/{len(centres)} CaImAn components have one of my peaks within {r} px")

The output above corresponds to the share of caiman detected peaks I also detect, and to the share of my detection CaImAn also detects, using the criterion of a peak falling whithin 3 pixels of a CaImAn component.

This was from tuning both the threshold and the window size by hand, but no tuning of any other parameters: I originally set them to values that seemed reasonible and didn't touch them.

In [ ]:
# Activation movie with a live agreement strip underneath (streamed frame by frame, low RAM).
# Top left: my activation movie. Top right: CaImAn's A·C(t). My rise detections in red on both.
# Bottom: per-frame counts (smoothed ~0.3 s for display) with a cursor at the current frame:
#   grey = CaImAn cells in a rising phase (C rises >15% of its max within 5 frames),
#   red = my distinct active zones, blue fill = CaImAn rising cells that contain one of my detections (next 11 frames).
from pathlib import Path
import scipy.sparse as sp
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

def activity_counts(rise, A, C, dims, frac=0.15, win=11):
    T, K = rise.shape[0], A.shape[1]
    mine = np.array([ndimage.label(r)[1] for r in rise])                          # distinct 2-D zones per frame
    Cp = np.pad(C, ((0, 0), (0, 5)), mode='edge')
    rising = ((Cp[:, 5:5 + T] - Cp[:, :T]) > frac * C.max(1, keepdims=True)).T    # (T, K)
    Ad = A.toarray()
    masks = sp.csr_matrix((Ad > 0.2 * Ad.max(0)).reshape(*dims, K, order='F').reshape(-1, K).astype(np.float32))
    hit = (sp.csr_matrix(rise.reshape(T, -1).astype(np.float32)) @ masks).toarray() > 0   # (T, K): detection in footprint
    cs = np.vstack([np.zeros((1, K)), np.cumsum(hit, 0)])
    hit_ahead = (cs[np.minimum(np.arange(T) + win, T)] - cs[:T]) > 0             # detection within the next `win` frames
    return mine, rising.sum(1), (rising & hit_ahead).sum(1)

def counts_strip(mine, caiman, both, n_frames, fps=29.78, width=600, height=160, smooth=9):
    """Render the count plot once as an RGB image; return it and the cursor x-pixel of every frame."""
    t = np.arange(len(mine)) / fps
    sm = lambda x: ndimage.uniform_filter1d(x.astype(float), smooth)
    fig = Figure(figsize=(width / 100, height / 100), dpi=100, facecolor='k'); FigureCanvasAgg(fig)
    ax = fig.add_axes([0.06, 0.2, 0.92, 0.75], facecolor='k')
    ax.fill_between(t, sm(both), color='deepskyblue', alpha=.6, lw=0, label='both')
    ax.plot(t, sm(caiman), color='0.75', lw=.8, label='CaImAn rising cells')
    ax.plot(t, sm(mine), color='red', lw=.8, label='my active zones')
    ax.set_xlim(0, (n_frames - 1) / fps); ax.set_ylim(0, None)
    ax.tick_params(colors='w', labelsize=6); [s.set_color('0.5') for s in ax.spines.values()]
    ax.set_xlabel('time (s)', color='w', fontsize=6, labelpad=1)
    ax.legend(frameon=False, ncol=3, fontsize=6, labelcolor='w', loc='upper right')
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    xs = ax.transData.transform(np.c_[np.arange(n_frames) / fps, np.zeros(n_frames)])[:, 0].round().astype(int)
    return img, xs

def activation_movie_counts(path, act, rise, A, C, dims, fps=30, trail=0.85):
    T = len(act)
    strip, xs = counts_strip(*activity_counts(rise, A, C, dims), n_frames=T)
    lo, hi = np.percentile(act[::10] ** .5, [1, 99.9])
    A = A.multiply(1 / A.max(0).toarray()).tocsc()                 # each footprint scaled to peak 1
    C = C / C.max(1, keepdims=True)                                # each trace scaled to peak 1
    red = np.zeros(act.shape[1:], np.float32)
    with iio.imopen(path, "w", plugin="pyav") as f:
        f.init_video_stream("libx264", fps=fps)
        for t in range(T):
            g = np.clip((act[t] ** .5 - lo) / (hi - lo), 0, 1)
            ac = np.clip((A @ C[:, t]).reshape(act.shape[1:], order='F'), 0, 1)
            red = np.maximum(red * trail, rise[t - 1]) if 0 < t <= len(rise) else red * trail
            r = ndimage.grey_dilation(red, size=3)
            panels = []
            for base in (g, ac):                                      # same red marks on both panels
                rgb = np.repeat(base[..., None], 3, axis=2)
                rgb[..., 1:] *= (1 - r[..., None])                     # green/blue down where detected...
                rgb[..., 0] = np.maximum(rgb[..., 0], r)               # ...red up
                panels.append(rgb)
            top = (np.concatenate(panels, axis=1) * 255).astype(np.uint8)
            cv2.putText(top, f"{t / 29.78:5.1f} s", (5, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            bottom = strip.copy()
            cv2.line(bottom, (int(xs[t]), 0), (int(xs[t]), bottom.shape[0] - 1), (255, 255, 255), 1)
            f.write_frame(np.ascontiguousarray(np.concatenate([top, bottom], axis=0)))

Path("out/S1").mkdir(parents=True, exist_ok=True)
activation_movie_counts("out/S1/activations_counts.mp4", act_mc, rise, cnm.estimates.A, cnm.estimates.C, dims)
Video("out/S1/activations_counts.mp4")

The match isn't perfect, and i'ts to be expected. My goal there was to start handling the data and understand the fundamental features of the measurement, as well as getting an idea of how a processing chain must handle noise and uncertainty. Overall, for a quick, naive processing pipeline cobbled together in a couple days after seeing calcium channel imaging data for the first time, I'm pretty happy with the results. I just won't claim it's a production level pipeline, because it isn't, and it was only possible because i had a CaImAn run as a reference to set thresholds until the results looked close enough.

The discrepencies are mostly a story of what the little pipeline I built is _for_. Real cells overlap somewhet (the camera does pick up some off axis light, actual neurons are a 3D structure and may lie on different planes), and way the signature of an activation is encoded in the detection pipeline is crude. It's pixel wise on stabilised images, only looking at groups of pixels rising together, while real world neurons are blobs with shapes stable in time, and a more refined method would take the time signal of an activation (rapid rise, exponential decay), and try to find the pixels that light up together following this profile.

From what I read of CaImAn's documentation, it seems to be what this software does under the hood, which explains from my own point of view why the runs tak so long: computing correlations on a 300 x 300 x 1000 data cube does take quite a lot of computation.

## Agreement with CaImAn, in time and in space

Counted against a chance level: the same detections shifted at random in time, or dropped at random positions in the lens circle.

In [ ]:
# Agreement between my rise detector and CaImAn, in time and in space, each against a chance level.
# Time: for each CaImAn cell, my onsets = first frame of a run where `rise` touches its footprint (A > 0.2 max);
#       CaImAn onsets = C rises by >15% of that cell's max within 5 frames (~170 ms).
#       A match is my onset within -2..+10 frames of a CaImAn onset (my erosion delays onsets by a few frames).
#       Chance: my onsets circularly shifted in time, same counts.
# Space: pk_rise points within 3 px of a CaImAn centre; chance: the same number of points dropped at random in the lens circle.
import scipy.sparse as sp
from scipy.spatial import cKDTree

def rise_vs_caiman(rise, A, C, dims, lag=(-2, 10), frac=0.15, n_null=200, seed=0):
    T, K = rise.shape[0], A.shape[1]
    Ad = A.toarray()
    masks = sp.csr_matrix((Ad > 0.2 * Ad.max(0)).reshape(*dims, K, order='F').reshape(-1, K).astype(np.float32))
    touch = ((sp.csr_matrix(rise.reshape(T, -1).astype(np.float32)) @ masks).toarray() > 0).T   # (K, T)
    mine = np.diff(touch.astype(np.int8), axis=1, prepend=0) == 1
    Cp = np.pad(C, ((0, 0), (0, 5)), mode='edge')
    up = ndimage.binary_closing(Cp[:, 5:5 + T] - Cp[:, :T] > frac * C.max(1, keepdims=True), structure=np.ones((1, 6), bool))
    ref = np.diff(up.astype(np.int8), axis=1, prepend=0) == 1

    def match(mine, ref):   # -> (my onsets matched, my onsets, CaImAn onsets matched, CaImAn onsets, cells caught)
        out = np.zeros(5, int)
        for o_k, e_k in zip(mine, ref):
            o, e = np.flatnonzero(o_k), np.flatnonzero(e_k)
            out[1] += len(o); out[3] += len(e)
            if len(o) and len(e):
                d = o[:, None] - e[None, :]
                ok = (d >= lag[0]) & (d <= lag[1])
                out[0] += ok.any(1).sum(); out[2] += ok.any(0).sum(); out[4] += ok.any()
        return out

    m = match(mine, ref)
    rng = np.random.default_rng(seed)
    null = np.array([match(np.roll(mine, rng.integers(60, T - 60), axis=1), ref) for _ in range(n_null)])
    print(f"time, lag {lag[0]}..{lag[1]} frames:")
    print(f"  precision {m[0]}/{m[1]} = {m[0]/m[1]:.0%} of my onsets fall on a CaImAn onset   (chance {np.mean(null[:, 0]/null[:, 1]):.0%})")
    print(f"  recall    {m[2]}/{m[3]} = {m[2]/m[3]:.0%} of CaImAn onsets have one of mine   (chance {np.mean(null[:, 2]/null[:, 3]):.0%})")
    print(f"  cells with at least one onset caught: {m[4]}/{ref.any(1).sum()}")
    return mine, ref

def pk_vs_caiman(pk, centres, fov, r=3, n_null=200, seed=0):
    rng = np.random.default_rng(seed)
    inside = np.argwhere(fov)
    def score(p):
        return (cKDTree(centres).query(p)[0] <= r).mean(), (cKDTree(p).query(centres)[0] <= r).mean()
    prec, rec = score(pk)
    null = np.array([score(inside[rng.choice(len(inside), len(pk), replace=False)]) for _ in range(n_null)])
    print(f"space, {r} px radius:")
    print(f"  precision {prec:.0%} of my {len(pk)} pk_rise points are near a CaImAn centre   (chance {null[:, 0].mean():.0%})")
    print(f"  recall    {rec:.0%} of {len(centres)} CaImAn centres have one of my points nearby   (chance {null[:, 1].mean():.0%})")

mine_on, ref_on = rise_vs_caiman(rise, cnm.estimates.A, cnm.estimates.C, dims)
pk_vs_caiman(pk_rise, cm.base.rois.com(cnm.estimates.A, *dims), fov)

# The 8 cells where I detect the most onsets: CaImAn trace, its onsets (grey) and mine (red).
C = cnm.estimates.C
top = np.argsort(-mine_on.sum(1))[:8]
fig, ax = plt.subplots(figsize=(10, 7))
t = np.arange(mine_on.shape[1]) / 29.78
for i, k in enumerate(top):
    ax.plot(t, C[k, :len(t)] / C[k].max() + 1.3 * i, 'k', lw=0.8)
    ax.plot(t[ref_on[k]], np.full(ref_on[k].sum(), 1.3 * i - 0.12), '|', color='0.5', ms=7)
    ax.plot(t[mine_on[k]], np.full(mine_on[k].sum(), 1.3 * i - 0.25), '|', color='red', ms=7)
ax.set_yticks(1.3 * np.arange(len(top)), [f"cell {k}" for k in top])
ax.set_xlabel("time (s)")
ax.set_title("CaImAn trace C (black), CaImAn onsets (grey), my onsets in the cell's footprint (red)")
fig.tight_layout()

Here again, this comparison tells the same story as before: my toy pipeline does better than chance, but doesn't beat CaImAn wielded by domain experts.

In [ ]:
import streaming
peak_s, onsets_s, noise_s, starts = streaming.run_session(files, "out/session_S1")
act_s  = np.load("out/session_S1/act.npy",  mmap_mode="r")   # (8935, 300, 300), no RAM cost until you slice
rise_s = np.load("out/session_S1/rise.npy", mmap_mode="r")
plt.imshow(peak_s ** .5, cmap="Greys_r")

In [ ]:
events, n_ev = ndimage.label(rise)                        # 3-D: one label per event
slices = ndimage.find_objects(events)                     # bounding box of each event
footprints = [(events[s] == i + 1).any(axis=0) for i, s in enumerate(slices)]   # each within its box

In [ ]:
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components

def event_zones(rise, iou=0.5, min_events=2):
    T, H, W = rise.shape
    events, n_ev = ndimage.label(rise)                       # 3-D blobs: one label per event
    t, y, x = np.nonzero(events)
    lab = events[t, y, x] - 1
    del events
    F = sp.csr_matrix((np.ones(len(lab), np.float32), (lab, y * W + x)), shape=(n_ev, H * W))
    F.data[:] = 1                                            # footprint of each event (duplicates over time -> 1)
    area = np.asarray(F.sum(1)).ravel()
    inter = (F @ F.T).tocoo()                                # pixels shared by each pair of events
    union = area[inter.row] + area[inter.col] - inter.data
    keep = inter.data / union > iou
    G = sp.csr_matrix((np.ones(keep.sum()), (inter.row[keep], inter.col[keep])), shape=(n_ev, n_ev))
    n_z, ev_zone = connected_components(G, directed=False)   # zone = group of events with overlapping footprints
    Z = sp.csr_matrix((np.ones(n_ev), (np.arange(n_ev), ev_zone)), shape=(n_ev, n_z))
    n_events = np.asarray(Z.sum(0)).ravel()                  # events per zone
    counts = (Z.T @ F).tocsc()                               # (zone, pixel): how many of the zone's events cover the pixel
    best = np.asarray(counts.argmax(0)).ravel()
    covered = np.asarray(counts.max(0).todense()).ravel() > 0
    ok = n_events[best] >= min_events                        # zones seen only once are likely noise
    zmap = np.where(covered & ok, best + 1, 0).reshape(H, W)
    return zmap, n_events, ev_zone

In [ ]:
plt.imshow(event_zones(rise, iou=0.5, min_events=2)[0] > 0)

In [ ]:
# Zones from my rise events (events whose footprints overlap), their traces from act_mc, and the best-matching CaImAn cell.
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components

def event_zones(rise, iou=0.5, min_events=2):
    """Group 3-D rise events into zones: events whose footprints overlap (IoU > iou) belong to the same zone."""
    T, H, W = rise.shape
    events, n_ev = ndimage.label(rise)                       # 3-D blobs: one label per event
    t, y, x = np.nonzero(events)
    lab = events[t, y, x] - 1
    del events
    t_first = np.full(n_ev, T); np.minimum.at(t_first, lab, t)   # first frame of each event
    F = sp.csr_matrix((np.ones(len(lab), np.float32), (lab, y * W + x)), shape=(n_ev, H * W))
    F.data[:] = 1                                            # footprint of each event
    area = np.asarray(F.sum(1)).ravel()
    inter = (F @ F.T).tocoo()                                # pixels shared by each pair of events
    keep = inter.data / (area[inter.row] + area[inter.col] - inter.data) > iou
    G = sp.csr_matrix((np.ones(keep.sum()), (inter.row[keep], inter.col[keep])), shape=(n_ev, n_ev))
    n_z, ev_zone = connected_components(G, directed=False)
    Z = sp.csr_matrix((np.ones(n_ev), (np.arange(n_ev), ev_zone)), shape=(n_ev, n_z))
    n_events = np.asarray(Z.sum(0)).ravel()
    counts = (Z.T @ F).tocsc()                               # (zone, pixel): events of the zone covering the pixel
    best = np.asarray(counts.argmax(0)).ravel()
    covered = np.asarray(counts.max(0).todense()).ravel() > 0
    zmap = np.where(covered & (n_events[best] >= min_events), best + 1, 0).reshape(H, W)
    return zmap, n_events, ev_zone, t_first

zmap, n_events, ev_zone, t_first = event_zones(rise)
ids = np.unique(zmap[zmap > 0])                              # kept zones (labels in zmap are zone index + 1)
print(f"{len(ids)} zones seen at least twice, median area {np.median(np.bincount(zmap.ravel())[ids]):.0f} px")

# Zone traces: mean of act_mc over each zone's pixels (aperture photometry with my own apertures)
T = act_mc.shape[0]
Zm = sp.csr_matrix((zmap == ids[:, None, None]).reshape(len(ids), -1).astype(np.float32))
Zm = sp.diags(1 / np.asarray(Zm.sum(1)).ravel()) @ Zm        # each row averages its zone
traces = (Zm @ act_mc.reshape(T, -1).T).T                    # (T, zones)

# Best-overlapping CaImAn component for each zone (shared pixels / union, footprint masks A > 0.2 max)
A = cnm.estimates.A.toarray(); K = A.shape[1]
masks = (A > 0.2 * A.max(0)).reshape(*dims, K, order='F').reshape(-1, K).astype(np.float32)
Zb = (zmap == ids[:, None, None]).reshape(len(ids), -1).astype(np.float32)
shared = Zb @ masks
iou_zc = shared / (Zb.sum(1)[:, None] + masks.sum(0)[None, :] - shared)
match_k, match_iou = iou_zc.argmax(1), iou_zc.max(1)
print(f"zones overlapping a CaImAn footprint (IoU > 0.3): {(match_iou > 0.3).sum()}/{len(ids)}")

# The 8 most active zones: my trace (black), the matching CaImAn trace (grey), my event onsets (red)
C = cnm.estimates.C
top = np.argsort(-n_events[ids - 1])[:8]
t = np.arange(T) / 29.78
fig, ax = plt.subplots(figsize=(10, 7))
for i, z in enumerate(top):
    on = t_first[ev_zone == ids[z] - 1] + 1                  # rise[j] is the step j -> j+1
    ax.plot(t, C[match_k[z], :T] / C[match_k[z]].max() + 1.3 * i, color='0.6', lw=0.8)
    ax.plot(t, traces[:, z] / traces[:, z].max() + 1.3 * i, 'k', lw=0.8)
    ax.plot(t[on], np.full(len(on), 1.3 * i - 0.15), '|', color='red', ms=7)
ax.set_yticks(1.3 * np.arange(len(top)), [f"zone {ids[z]}\n(cell {match_k[z]}, IoU {match_iou[z]:.2f})" for z in top], fontsize=8)
ax.set_xlabel("time (s)")
ax.set_title("my zone trace from act_mc (black), best-matching CaImAn trace C (grey), my events (red)")
fig.tight_layout()

In [ ]:
# Map of the detections by both methods, on my activation map (max of act_mc over time).
# Coloured patches: my zones (one colour per zone). White: CaImAn footprint contours.
# Yellow: the 8 zones of the trace figure (labelled) and the CaImAn contour each one was matched to.
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(act_mc.max(0) ** .5, cmap='Greys_r')
lut = np.random.default_rng(0).integers(0, 20, zmap.max() + 1)          # shuffled colours, so neighbours differ
ax.imshow(np.ma.masked_equal(np.where(zmap > 0, lut[zmap], -1), -1), cmap='tab20', vmin=0, vmax=19,
          alpha=0.5, interpolation='nearest')
for c in conts:
    ax.plot(*c['coordinates'].T, color='w', lw=0.5, alpha=0.9)
for z in top:
    ax.plot(*conts[match_k[z]]['coordinates'].T, color='yellow', lw=1.2)
    y, x = ndimage.center_of_mass(zmap == ids[z])
    ax.text(x + 4, y - 4, str(ids[z]), color='yellow', fontsize=9, fontweight='bold')
allc = np.vstack([c['coordinates'] for c in conts]); allc = allc[np.isfinite(allc).all(1)]
(x0, y0), (x1, y1) = allc.min(0) - 10, allc.max(0) + 10          # crop to where the cells are
ax.set_xlim(x0, x1); ax.set_ylim(y1, y0); ax.set_axis_off()
ax.set_title(f"my {len(ids)} zones (colour) vs CaImAn's {len(conts)} footprints (white)")
fig.tight_layout()

## Full session, compared with the authors' published extraction

My streamed results over the whole session (5 min): the activation map, zones built from the rise events, and their traces (mean of `act` over each zone). The reference is the authors' CNMF-E extraction stored in the session NWB: accepted components only, footprints downsampled to my 300×300 grid, traces aligned by frame index (row k of their C = raw frame k).

The code in `fullsession.py` is boilerplate written with an AI assistant (Claude Code); the method is the one developed above.

In [ ]:
import fullsession
res = fullsession.compare_with_authors("out/session_S1", "data/sub-F06/sub-F06_ses-CPFE-F06-S1_behavior+image+ophys.nwb", title="S1: ")